<a href="https://colab.research.google.com/github/MilkySon/RL_carracing_competition/blob/261102/261101_E10_E13_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 261101 CarRacing: E10 → E11 → E12 → E13

**84×84 grayscale / stack 4 / skip 4**. 실험별 300,000 **원본** 프레임. 각 실험은 처음부터 독립 학습합니다. Colab GPU 런타임 권장.

- E10: DDQN만 적용하여 E3와 비교.
- E11: 동일 DDQN을 독립 학습하며 역주행 **진단 로그** 추가(알고리즘 개선 실험은 아님).
- E12: 음수 보상 경험 25% 별도 replay(역주행 검출과 동일하지 않음).
- E13: E12 + 학습 중에만 방향 보조 분류. **내부 시뮬레이터 레이블을 학습에 써도 되는지 규칙 확인 후 실행.**

결과: `MyDrive/261101carracing_results/E10` ... `E13`에 `best.pt`, `latest.pt`, `final.pt`, CSV, JSON, MP4, ONNX, 진단 로그를 분리 저장합니다. Colab 무료 GPU 사용량에 따라 전부 완료하는 데 여러 세션이 필요할 수 있습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, json, shutil, csv, statistics
ROOT=Path('/content/drive/MyDrive/261101carracing_competition')
# ZIP을 업로드했다면 같은 위치에 압축을 풀어주세요.
if not (ROOT/'train.py').exists():
    import zipfile
    archive=Path('/content/drive/MyDrive/261101carracing_competition.zip')
    assert archive.exists(), f'Missing folder or zip: {ROOT} / {archive}'
    with zipfile.ZipFile(archive) as z:z.extractall('/content/drive/MyDrive')
assert (ROOT/'train.py').exists(), ROOT
os.chdir(ROOT)
RESULTS=Path('/content/drive/MyDrive/261101carracing_results')
RESULTS.mkdir(parents=True,exist_ok=True)
print('Code:',ROOT,'Results:',RESULTS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code: /content/drive/MyDrive/261101carracing_competition Results: /content/drive/MyDrive/261101carracing_results


In [ ]:
%pip -q install swig
%pip -q install -r requirements.txt
%pip -q install moviepy imageio-ffmpeg onnx onnxruntime
import torch
print('GPU:',torch.cuda.is_available())

GPU: False


## 실행 설정
`RUN_EXPERIMENTS`를 이용하면 중간 실험부터 재개할 수 있습니다. **재실행 시 기존 실험 폴더가 있으면 학습을 덮어쓰지 않습니다.** `latest.pt`에서 재개하는 경우 replay 버퍼는 복원되지 않으므로 엄밀한 재현성은 없습니다.

E13은 `ALLOW_PRIVILEGED_TRAINING_LABELS=True`로 명시적으로 승인해야 실행됩니다.

In [ ]:
RUN_EXPERIMENTS=['E10','E11','E12','E13']
ALLOW_PRIVILEGED_TRAINING_LABELS=False  # 대회 규칙 확인 후 True
BUDGET=300_000
TRAIN_SEED=0
EVAL_SEEDS='1000-1009'
VIDEO_SEEDS='1000-1009'  # 저장 공간 절약; 필요하면 1000-1009
EXPS={'E10':'ddqn_e10','E11':'ddqn_e11','E12':'ddqn_e12','E13':'ddqn_e13'}

def run_cmd(argv,log):
    print('\n$', ' '.join(map(str,argv)))
    with open(log,'a',encoding='utf-8') as f:
        proc=subprocess.Popen(list(map(str,argv)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='');f.write(line);f.flush()
        rc=proc.wait()
    if rc:raise RuntimeError(f'Command failed ({rc}): {argv}; log={log}')

def execute_experiment(e):
    assert e in EXPS
    if e=='E13' and not ALLOW_PRIVILEGED_TRAINING_LABELS:
        print('E13 skipped: set ALLOW_PRIVILEGED_TRAINING_LABELS=True only after rule check');return
    d=RESULTS/e;d.mkdir(parents=True,exist_ok=True)
    for folder in ['videos','onnx','diagnostics','reports']: (d/folder).mkdir(exist_ok=True)
    agent=EXPS[e]
    latest=d/'latest.pt';final=d/'final.pt';best=d/'best.pt'
    if not final.exists():
        cmd=['python','train.py','--agent',agent,'--seed',TRAIN_SEED,'--budget',BUDGET,'--eval-every',50000,'--run-dir',d]
        if latest.exists():cmd.append('--resume')
        run_cmd(cmd,d/'training_console.log')
    else:print(e,'training already completed')
    assert best.exists(), f'{e}: best.pt missing'
    model=d/'onnx'/f'{e}.onnx'
    if not model.exists():run_cmd(['python','export.py','--ckpt',best,'--out',model],d/'export_console.log')
    score=d/'reports'/'scores_1000_1009.json'
    if not score.exists():run_cmd(['python','evaluate.py','--ckpt',best,'--seeds',EVAL_SEEDS,'--out',score],d/'evaluation_console.log')
    # Evaluate with video only a few seeds to avoid huge files.
    if not list((d/'videos').glob('*.mp4')):
        run_cmd(['python','evaluate.py','--ckpt',best,'--seeds',VIDEO_SEEDS,'--video',d/'videos'],d/'video_console.log')
    # Direction diagnostics are observational; actions are always produced by DDQN.
    diag=d/'diagnostics'/'direction_1000_1009.csv'
    if not diag.exists():run_cmd(['python','diagnose_direction.py','--ckpt',best,'--seeds',EVAL_SEEDS,'--out',diag],d/'diagnostic_console.log')
    print('FINISHED',e,'->',d)

## E10: Double DQN 단독

E3와 비교하기 위해 알고리즘만 DDQN으로 변경합니다.

In [ ]:
if 'E10' in RUN_EXPERIMENTS:
    execute_experiment('E10')


$ python train.py --agent ddqn_e10 --seed 0 --budget 300000 --eval-every 50000 --run-dir /content/drive/MyDrive/261101carracing_results/E10
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
agent=ddqn_e10 action=discrete device=cpu budget=300,000 frames
[    1,000] ep    1 return   -34.6 (79 fps) 
[    2,000] ep    2 return   -58.6 (74 fps) 
[    3,000] ep    3 return   -36.8 (76 fps) 
[    4,000] ep    4 return   -60.9 (77 fps) 
[    5,000] ep    5 return   -34.2 (77 fps) 
[    6,000] ep    6 return   -16.3 (78 fps) 
[    7,000] ep    7 return   -36.2 (79 fps) 
[    8,000] ep    8 return   -31.7 (79 fps) 
[    9,000] ep    9 return   -37.3 (79 fps) 
[   10,000] ep   10 retur

## E11: Double DQN + 역주행 진단

E10과 같은 학습 알고리즘이며, 방향/역주행을 별도 CSV로 관찰합니다. 독립적인 개선 방법으로 해석하면 안 됩니다.

In [ ]:
if 'E11' in RUN_EXPERIMENTS:
    execute_experiment('E11')


$ python train.py --agent ddqn_e11 --seed 0 --budget 300000 --eval-every 50000 --run-dir /content/drive/MyDrive/261101carracing_results/E11
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
agent=ddqn_e11 action=discrete device=cpu budget=300,000 frames
[    1,000] ep    1 return   -34.6 (87 fps) 
[    2,000] ep    2 return   -58.6 (80 fps) 
[    3,000] ep    3 return   -36.8 (79 fps) 
[    4,000] ep    4 return   -60.9 (79 fps) 
[    5,000] ep    5 return   -34.2 (79 fps) 
[    6,000] ep    6 return   -16.3 (79 fps) 
[    7,000] ep    7 return   -36.2 (79 fps) 
[    8,000] ep    8 return   -31.7 (80 fps) 
[    9,000] ep    9 return   -37.3 (80 fps) 
[   10,000] ep   10 retur

## E12: 음수 보상 경험 Replay

음수 보상 transition을 25% 섞습니다. 이는 실제 역주행 레이블이 아닌 실패의 대리 지표입니다.

In [ ]:
if 'E12' in RUN_EXPERIMENTS:
    execute_experiment('E12')


$ python train.py --agent ddqn_e12 --seed 0 --budget 300000 --eval-every 50000 --run-dir /content/drive/MyDrive/261101carracing_results/E12 --resume
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
resume from /content/drive/MyDrive/261101carracing_results/E12/latest.pt (frames=100001, episodes=106)
agent=ddqn_e12 action=discrete device=cpu budget=300,000 frames
[  101,001] ep  107 return   176.2 (111 fps) 
[  101,461] ep  108 return   -84.6 (115 fps) 
[  101,983] ep  109 return    66.4 (117 fps) 
[  102,983] ep  110 return    70.8 (119 fps) 
[  103,983] ep  111 return   250.4 (116 fps) 
[  104,597] ep  112 return   231.9 (117 fps) 
[  104,900] ep  113 return    29.2 (117 fp

## E13: 학습용 방향 보조 과제

시뮬레이터 트랙 접선과 차량 속도로 만든 방향 레이블을 **학습에만** 사용합니다. 규칙 검토 전에는 기본적으로 건너뜁니다.

In [ ]:
if 'E13' in RUN_EXPERIMENTS:
    execute_experiment('E13')

E13 skipped: set ALLOW_PRIVILEGED_TRAINING_LABELS=True only after rule check


## E14: 음수 보상 경험 Replay (seed 42)

음수 보상 transition을 25% 섞습니다. 이는 실제 역주행 레이블이 아닌 실패의 대리 지표입니다.

E12와 동일한 알고리즘.

하지만 seed = 42로 설정해두어 보상 transition의 비율이 25%가 성능 저하의 원인인지, 아니면 다른게 원인인지 알아보고자 합니다.

In [ ]:

# =====================================================
# E14: DDQN + Negative Reward Replay 25%
#      TRAIN_SEED = 42
# =====================================================

from pathlib import Path
import json

E14_SEED = 42
E14_BUDGET = 300_000
E14_AGENT = "ddqn_e12"  # E12와 동일한 알고리즘

E14_DIR = RESULTS / "E14"

for folder in ["videos", "onnx", "diagnostics", "reports"]:
    (E14_DIR / folder).mkdir(parents=True, exist_ok=True)

best = E14_DIR / "best.pt"
final = E14_DIR / "final.pt"
latest = E14_DIR / "latest.pt"

# 1. 독립 학습
# 기존 checkpoint에서 resume하지 않음
if not final.exists():
    if latest.exists():
        raise RuntimeError(
            "E14/latest.pt가 존재합니다. "
            "독립적인 처음부터 학습을 위해 기존 파일을 "
            "확인하세요. 자동 resume하지 않습니다."
        )

    run_cmd([
        "python", "train.py",
        "--agent", E14_AGENT,
        "--seed", E14_SEED,
        "--budget", E14_BUDGET,
        "--eval-every", 50000,
        "--run-dir", E14_DIR
    ], E14_DIR / "training_console.log")
else:
    print("E14 training already completed")

assert best.exists(), "E14/best.pt가 없습니다."

# 2. ONNX 변환
onnx_path = E14_DIR / "onnx" / "E14.onnx"

if not onnx_path.exists():
    run_cmd([
        "python", "export.py",
        "--ckpt", best,
        "--out", onnx_path
    ], E14_DIR / "export_console.log")

# 3. E10~E12와 동일한 평가 시드
score_path = E14_DIR / "reports" / "scores_1000_1009.json"

if not score_path.exists():
    run_cmd([
        "python", "evaluate.py",
        "--ckpt", best,
        "--seeds", "1000-1009",
        "--out", score_path
    ], E14_DIR / "evaluation_console.log")

# 4. 주행 영상
if not list((E14_DIR / "videos").glob("*.mp4")):
    run_cmd([
        "python", "evaluate.py",
        "--ckpt", best,
        "--seeds", "1000-1002",
        "--video", E14_DIR / "videos"
    ], E14_DIR / "video_console.log")

# 5. 역주행 진단
diag_path = (
    E14_DIR / "diagnostics" /
    "direction_1000_1009.csv"
)

if not diag_path.exists():
    run_cmd([
        "python", "diagnose_direction.py",
        "--ckpt", best,
        "--seeds", "1000-1009",
        "--out", diag_path
    ], E14_DIR / "diagnostic_console.log")

# 6. E12 vs E14 비교
import pandas as pd
import statistics

rows = []

for exp in ["E10", "E12", "E14"]:
    path = RESULTS / exp / "reports" / "scores_1000_1009.json"
    if not path.exists():
        continue

    obj = json.loads(path.read_text())
    scores = obj["returns"]

    rows.append({
        "experiment": exp,
        "mean": round(statistics.mean(scores), 2),
        "std": round(statistics.pstdev(scores), 2),
        "min": round(min(scores), 2),
        "max": round(max(scores), 2),
        "over_900": sum(x >= 900 for x in scores)
    })

comparison = pd.DataFrame(rows)
display(comparison)

comparison.to_csv(
    RESULTS / "E14" / "comparison.csv",
    index=False
)

print("E14 FINISHED:", E14_DIR)



$ python train.py --agent ddqn_e12 --seed 42 --budget 300000 --eval-every 50000 --run-dir /content/drive/MyDrive/261101carracing_results/E14
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
agent=ddqn_e12 action=discrete device=cpu budget=300,000 frames
[    1,000] ep    1 return   -63.6 (120 fps) 
[    2,000] ep    2 return   -58.5 (121 fps) 
[    3,000] ep    3 return   -43.8 (127 fps) 
[    4,000] ep    4 return   -40.8 (125 fps) 
[    5,000] ep    5 return   -58.6 (124 fps) 
[    6,000] ep    6 return   -21.0 (124 fps) 
[    7,000] ep    7 return    -2.3 (121 fps) 
[    8,000] ep    8 return   -60.7 (119 fps) 
[    9,000] ep    9 return   -43.0 (118 fps) 
[   10,000] ep 

,experiment,mean,std,min,max,over_900
0,E10,893.20,19.91,857.14,922.0,4
1,E12,792.96,185.11,348.72,913.5,5
2,E14,813.57,159.41,358.33,912.1,5


E14 FINISHED: /content/drive/MyDrive/261101carracing_results/E14


## 결과 요약 CSV 및 GitHub 공유 준비
실험 종료 후 로컬 보고서를 생성합니다. **GitHub에 자동 게시하지 않습니다.** GitHub 공유 셀은 사용자가 직접 실행하고 인증해야 합니다. 모델 `.onnx` 및 `.pt`는 용량 제한에 따라 Git LFS 또는 Release/Drive 링크가 필요할 수 있습니다.

In [ ]:
import pandas as pd
rows=[]
for e in EXPS:
    d=RESULTS/e;f=d/'reports'/'scores_1000_1009.json'
    if not f.exists():continue
    obj=json.loads(f.read_text()); vals=obj.get('returns',[])
    if not vals:continue
    rows.append({'experiment':e,'mean':float(statistics.mean(vals)),'std':float(statistics.pstdev(vals)),'min':float(min(vals)),'max':float(max(vals)),'onnx':str(d/'onnx'/f'{e}.onnx')})
summary=pd.DataFrame(rows)
summary.to_csv(RESULTS/'summary.csv',index=False)
display(summary)
report=['# E10–E13 experimental results','', 'Same train budget: 300,000 raw frames per experiment. Public evaluation seeds 1000–1009.','',summary.to_markdown(index=False) if len(summary) else 'No completed experiments yet.','', 'E11 is diagnostic-only, not an algorithmic change. E12 uses negative reward as a failure proxy. E13 uses simulator-derived train-only direction labels.']
(RESULTS/'README_results.md').write_text('\n'.join(report),encoding='utf-8')

### GitHub 공유 (선택 실행)
권장: GitHub 저장소의 별도 브랜치에 결과 CSV, Markdown, 코드 변경분, 작은 영상만 올리고, 큰 `.pt`/`.onnx`는 Drive에 보관하세요. 아래 셀은 **기본 비활성화**입니다. 개인 액세스 토큰을 노트북에 하드코딩하지 마세요.

In [ ]:
PUBLISH_TO_GITHUB=False
if PUBLISH_TO_GITHUB:
    from google.colab import userdata
    import getpass
    token=getpass.getpass('GitHub fine-grained PAT (repo contents write): ')
    assert token
    # clone into Colab ephemeral storage; token not saved in remote URL
    repo=Path('/content/rl_carracing_results_publish')
    if repo.exists():shutil.rmtree(repo)
    env=os.environ.copy()
    env['GIT_ASKPASS']='/bin/false'
    import base64
    auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    subprocess.run(['git','-c',f'http.extraHeader=Authorization: Basic {auth}','clone','https://github.com/MilkySon/RL_carracing_competition.git',str(repo)],check=True)
    subprocess.run(['git','checkout','-B','experiments-261101'],cwd=repo,check=True)
    dst=repo/'results'/'261101';dst.mkdir(parents=True,exist_ok=True)
    shutil.copy2(RESULTS/'summary.csv',dst/'summary.csv')
    shutil.copy2(RESULTS/'README_results.md',dst/'README.md')
    for e in EXPS:
        src=RESULTS/e
        if not src.exists():continue
        out=dst/e;out.mkdir(exist_ok=True)
        for name in ['train.csv','eval.csv','config.json']:
            if (src/name).exists():shutil.copy2(src/name,out/name)
        for rel in ['reports/scores_1000_1009.json','diagnostics/direction_1000_1009.csv']:
            f=src/rel
            if f.exists(): (out/rel).parent.mkdir(parents=True,exist_ok=True);shutil.copy2(f,out/rel)
    subprocess.run(['git','add','results/261101'],cwd=repo,check=True)
    subprocess.run(['git','-c','user.name=Colab RL Experiments','-c','user.email=colab@example.invalid','commit','-m','Add E10-E13 evaluation reports'],cwd=repo,check=True)
    subprocess.run(['git','-c',f'http.extraHeader=Authorization: Basic {auth}','push','-u','origin','experiments-261101'],cwd=repo,check=True)
    token=None;auth=None
    print('Branch published: https://github.com/MilkySon/RL_carracing_competition/tree/experiments-261101')
else:print('Publishing disabled. Set PUBLISH_TO_GITHUB=True and execute this cell to publish results.')